In [ ]:
! pip install kagglehub

In [ ]:
import kagglehub

In [ ]:
path = kagglehub.dataset_download("jjinho/wikipedia-20230701")
print("Path to dataset files:", path)

In [ ]:
!rm -rf /content/drive/MyDrive/Activity/dl-HybridRAG/kaggleDatasets/wikipedia-20230701

In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ /content/drive/MyDrive/Activity/dl-HybridRAG/kaggle

In [ ]:
!rm -rf /root/.cache/kagglehub/datasets/jjinho/wikipedia-20230701/versions/2

In [ ]:
!pip install pyarrow

# Analisi parquet

In [ ]:
import os
import shutil
import re
from datetime import datetime
from pathlib import Path
from typing import Union

import pandas as pd

In [ ]:
# Percorso al tuo file .parquet (modifica con il tuo)
file_path = '/content/drive/MyDrive/Activity/dl-HybridRAG/kaggle/datasets/jjinho/wikipedia-20230701/versions/2/a.parquet'

# Legge il file Parquet in un DataFrame
df = pd.read_parquet(file_path)

# Mostra le prime righe
print(df.head())

In [ ]:
print(df.columns)         # Mostra tutte le colonne
print(df.shape)           # Dimensione (righe, colonne)
print(df.dtypes)          # Tipi di dato

In [ ]:
for i in range(3):
  print(df.loc[i, ['title']].values[0]+'\n')
  print(df.loc[i, ['categories']].values[0]+'\n')
  print(df.loc[i, ['text']].values[0]+'\n')
  print('\n\n')

In [ ]:
class Logger:
    def __init__(self, folder_path: Union[str, Path], filename: str):
        """
        Crea un logger che scrive su folder_path/filename.
        Se folder_path non esiste, lo crea.
        Se filename esiste, lo rinomina in filename-pre-<timestamp>.<ext>
        """
        self.folder_path = Path(folder_path)
        # Crea la cartella se non esiste
        if not self.folder_path.exists():
            print(f"Logger: creando directory di log '{self.folder_path}'")
            self.folder_path.mkdir(parents=True, exist_ok=True)
        else:
            print(f"Logger: directory di log già esistente '{self.folder_path}'")

        self.filename = filename
        self.filepath = self.folder_path / filename

        # Rinomina il file esistente, se presente
        if self.filepath.exists():
            print(f"Logger: file di log esistente trovato '{self.filepath}', rinominazione in corso...")
            ts = datetime.now().strftime("%Y%m%dT%H%M%S")
            name, ext = os.path.splitext(self.filename)
            backup_name = f"{name}-pre-{ts}{ext}"
            backup_path = self.folder_path / backup_name
            # Rinomino il file esistente
            self.filepath.rename(backup_path)
            print(f"Logger: backup creato '{backup_path}'")
        else:
            print(f"Logger: nessun file di log esistente trovato, ne verrà creato uno nuovo")

        # Apre il nuovo file di log in append
        self._file = open(self.filepath, "a", encoding="utf-8")
        print(f"Logger: nuovo file di log aperto '{self.filepath}'")

    def log(self, message: str):
        """
        Scrive una riga di log preceduta da timestamp ISO e stampa in console.
        """
        now = datetime.now().isoformat(sep=" ", timespec="seconds")
        log_entry = f"[{now}] {message}"
        # Scrive su file di log
        self._file.write(f"{log_entry}\n")
        self._file.flush()
        # Stampa anche in console
        print(log_entry)

    def close(self):
        """
        Chiude il file di log.
        """
        if not self._file.closed:
            self._file.close()
            print(f"Logger: file di log chiuso '{self.filepath}'")

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


In [ ]:
# chiedi all’utente dove loggare e con che nome
folder = "/content/drive/MyDrive/Activity/dl-HybridRAG/logs"
filename = "wikiSeparators.txt"

In [ ]:
logger = Logger(folder_path=folder,filename=filename)

In [ ]:
def extract_match_descriptions(
    df: pd.DataFrame,
    logger: Logger,
    verbosity: int = 0
) -> pd.DataFrame:
    """
    Estrae descrizioni tra match di pattern '==...==' in ogni text del DataFrame.

    Parametri:
    - df: DataFrame con colonna 'text'
    - logger: istanza di Logger per logging
    - verbosity: 0 = log dettagliati, 1 = solo errori

    Ritorna:
    DataFrame con colonne ['match', 'desc', 'desc_len'], una riga per ogni match con la descrizione massima.
    """
    try:
        if verbosity == 0:
            logger.log(f"extract_match_descriptions: inizio elaborazione di {len(df)} righe")

        pattern = re.compile(r"==\w*(?:\s)?\w*(?:\s)?==")
        records = []  # lista intermedia di dict

        for idx, row in df.iterrows():
            text = row.get('text', '')
            if verbosity == 0:
                logger.log(f"Riga {idx}: analizzo text di lunghezza {len(text)}")

            matches = list(pattern.finditer(text))
            if not matches:
                if verbosity == 0:
                    logger.log(f"Riga {idx}: nessun match trovato, uso NOKEY")
                # Crea DataFrame con NOKEY per questo text
                df_nokey = pd.DataFrame([{
                    'match': 'NOKEY',
                    'desc': text,
                    'desc_len': len(text)
                }])
                return df_nokey

            if verbosity == 0:
                logger.log(f"Riga {idx}: trovati {len(matches)} match")

            for i, m in enumerate(matches):
                match_str = m.group()
                start = m.end()
                end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
                desc = text[start:end].strip()

                records.append({'match': match_str, 'desc': desc})

        # Costruisci DataFrame intermedio
        rec_df = pd.DataFrame(records)
        if verbosity == 0:
            logger.log(f"Intermedio: {len(rec_df)} record creati")

        # Mantieni per ogni match la descrizione più lunga
        rec_df['desc_len'] = rec_df['desc'].str.len()
        rec_df = rec_df.sort_values('desc_len', ascending=False)
        result_df = rec_df.drop_duplicates(subset='match', keep='first')
        result_df = result_df[['match', 'desc', 'desc_len']].reset_index(drop=True)

        if verbosity == 0:
            logger.log(f"Elaborazione completata: {len(result_df)} match unici")

        return result_df

    except Exception as e:
        logger.log(f"extract_match_descriptions: ERRORE → {e}")
        raise



In [ ]:
def process_parquet(
    file_path: str,
    logger: Logger,
    verbosity: int = 0
) -> pd.DataFrame:
    """
    Carica un file .parquet da file_path, applica extract_match_descriptions per ogni record,
    e restituisce un DataFrame con le colonne ['match', 'desc', 'desc_len'],
    rimuovendo duplicati mantenendo la descrizione più lunga per ogni match.

    Parametri:
    - file_path: percorso al file .parquet
    - logger: istanza di Logger per logging
    - verbosity: 0 = log dettagliati, 1 = solo errori

    Ritorna:
    DataFrame aggregato con colonne ['match', 'desc', 'desc_len']
    """
    try:
        if verbosity == 0:
            logger.log(f"process_parquet: caricamento file {file_path}")
        # Leggi file .parquet
        df = pd.read_parquet(file_path)
        if verbosity == 0:
            logger.log(f"process_parquet: {len(df)} record letti dal file")
        # DataFrame intermedio
        total = len(df)
        next_pct = 1
        # Log iniziale 0% del documento analizzato
        logger.log(f"process_parquet: 0% del documento analizzato")
        # Processa ogni record
        intermediate_df = pd.DataFrame(columns=['match','desc','desc_len'])
        for idx, row in df.iterrows():
            # Estrai match per il singolo record
            single_df = pd.DataFrame([row])
            extracted = extract_match_descriptions(single_df, logger, verbosity)
            # Aggiungi al DataFrame intermedio
            intermediate_df = pd.concat([intermediate_df, extracted], ignore_index=True)
            # Log progresso ogni 1% se è il prossimo punto
            pct = (idx + 1) * 100 // total
            if pct >= next_pct:
                logger.log(f"process_parquet: {pct}% del documento analizzato")
                next_pct = pct + 1
            single_df = pd.DataFrame([row])
            extracted = extract_match_descriptions(single_df, logger, verbosity)
            # Aggiungi al DataFrame intermedio
            intermediate_df = pd.concat([intermediate_df, extracted], ignore_index=True)
        # Rimuovi duplicati mantenendo desc_len massimo
        intermediate_df = intermediate_df.sort_values('desc_len', ascending=False)
        result_df = intermediate_df.drop_duplicates(subset='match', keep='first').reset_index(drop=True)
        if verbosity == 0:
            logger.log(f"process_parquet: risultato finale contiene {len(result_df)} match unici")
        return result_df
    except Exception as e:
        logger.log(f"process_parquet: ERRORE → {e}")
        raise

In [ ]:
def process_parquet_mocked(
    file_path: str,
    logger: Logger,
    verbosity: int = 0
) -> pd.DataFrame:
    """
    Carica un file .parquet da file_path, applica extract_match_descriptions per ogni record,
    e restituisce un DataFrame con le colonne ['match', 'desc', 'desc_len'],
    rimuovendo duplicati mantenendo la descrizione più lunga per ogni match.

    Parametri:
    - file_path: percorso al file .parquet
    - logger: istanza di Logger per logging
    - verbosity: 0 = log dettagliati, 1 = solo errori

    Ritorna:
    DataFrame aggregato con colonne ['match', 'desc', 'desc_len']
    """
    try:
        if verbosity == 0:
            logger.log(f"process_parquet: caricamento file {file_path}")
        # Leggi file .parquet
        df = pd.read_parquet(file_path)
        if verbosity == 0:
            logger.log(f"process_parquet: {len(df)} record letti dal file")
        # DataFrame intermedio
        total = len(df)
        next_pct = 1
        # Log iniziale 0% del documento analizzato
        logger.log(f"process_parquet: 0% del documento analizzato")
        # Processa ogni record
        intermediate_df = pd.DataFrame(columns=['match','desc','desc_len'])
        for idx, row in df.iterrows():
            # Estrai match per il singolo record
            single_df = pd.DataFrame([row])
            extracted = extract_match_descriptions(single_df, logger, verbosity)
            # Aggiungi al DataFrame intermedio
            intermediate_df = pd.concat([intermediate_df, extracted], ignore_index=True)
            # Log progresso ogni 1% se è il prossimo punto
            pct = (idx + 1) * 100 // total
            if pct >= next_pct:
                logger.log(f"process_parquet: {pct}% del documento analizzato")
                next_pct = pct + 1
            single_df = pd.DataFrame([row])
            extracted = extract_match_descriptions(single_df, logger, verbosity)
            # Aggiungi al DataFrame intermedio
            intermediate_df = pd.concat([intermediate_df, extracted], ignore_index=True)
            break
        # Rimuovi duplicati mantenendo desc_len massimo
        intermediate_df = intermediate_df.sort_values('desc_len', ascending=False)
        result_df = intermediate_df.drop_duplicates(subset='match', keep='first').reset_index(drop=True)
        if verbosity == 0:
            logger.log(f"process_parquet: risultato finale contiene {len(result_df)} match unici")
        return result_df
    except Exception as e:
        logger.log(f"process_parquet: ERRORE → {e}")
        raise

In [ ]:
dfA = process_parquet_mocked(logger=logger,verbosity=0,file_path='/content/drive/MyDrive/Activity/dl-HybridRAG/kaggle/datasets/jjinho/wikipedia-20230701/versions/2/a.parquet')

In [ ]:
def elaborateDataset(
    logger: Logger,
    verbosity: int,
    dataset_path: str,
    filename_regex: str,
    output_folder: str
) -> None:
    """
    Elabora tutti i file .parquet in dataset_path il cui nome rispetta filename_regex,
    applica process_parquet per ciascuno e salva il DataFrame risultante in CSV in output_folder.

    Parametri:
    - logger: istanza di Logger per logging
    - verbosity: 0 = log dettagliati, 1 = solo errori
    - dataset_path: path alla directory contenente i file .parquet
    - filename_regex: regex per validare i nomi dei file (includendo estensione)
    - output_folder: path alla cartella dove salvare i file CSV di output
    """
    try:
        # Assicurati che i path esistano
        ds_path = Path(dataset_path)
        out_path = Path(output_folder)
        if not ds_path.exists():
            logger.log(f"elaborateDataset: directory dataset non esistente '{ds_path}', arresto")
            raise FileNotFoundError(f"Directory dataset non trovata: {ds_path}")
        else:
            logger.log(f"elaborateDataset: directory dataset esistente '{ds_path}'")
        if not out_path.exists():
            logger.log(f"elaborateDataset: directory output non esistente, la creo '{out_path}'")
            out_path.mkdir(parents=True, exist_ok=True)
        else:
            logger.log(f"elaborateDataset: directory output esistente '{out_path}'")

        # Trova e filtra i file .parquet
        pattern = re.compile(filename_regex)
        files = [f for f in ds_path.iterdir() if f.is_file() and pattern.match(f.name)]
        files.sort(key=lambda f: f.name)
        logger.log(f"elaborateDataset: trovati {len(files)} file validi")

        # Elaborazione file
        for file in files:
            logger.log(f"elaborateDataset: inizio elaborazione file '{file.name}'")
            df_res = process_parquet(str(file), logger, verbosity)
            # Costruisci nome file CSV con timestamp
            ts = datetime.now().strftime("%Y%m%dT%H%M%S")
            base = file.stem
            csv_name = f"{base}_{ts}.csv"
            csv_path = out_path / csv_name
            # Salva in CSV
            df_res.to_csv(csv_path, index=False)
            logger.log(f"elaborateDataset: salvato risultato in '{csv_path}'")

    except Exception as e:
        logger.log(f"elaborateDataset: ERRORE → {e}")
        raise

In [ ]:
def elaborateDatasetMocked(
    logger: Logger,
    verbosity: int,
    dataset_path: str,
    filename_regex: str,
    output_folder: str
) -> None:
    """
    Elabora tutti i file .parquet in dataset_path il cui nome rispetta filename_regex,
    applica process_parquet per ciascuno e salva il DataFrame risultante in CSV in output_folder.

    Parametri:
    - logger: istanza di Logger per logging
    - verbosity: 0 = log dettagliati, 1 = solo errori
    - dataset_path: path alla directory contenente i file .parquet
    - filename_regex: regex per validare i nomi dei file (includendo estensione)
    - output_folder: path alla cartella dove salvare i file CSV di output
    """
    try:
        # Assicurati che i path esistano
        ds_path = Path(dataset_path)
        out_path = Path(output_folder)
        if not ds_path.exists():
            logger.log(f"elaborateDataset: directory dataset non esistente '{ds_path}', arresto")
            raise FileNotFoundError(f"Directory dataset non trovata: {ds_path}")
        else:
            logger.log(f"elaborateDataset: directory dataset esistente '{ds_path}'")
        if not out_path.exists():
            logger.log(f"elaborateDataset: directory output non esistente, la creo '{out_path}'")
            out_path.mkdir(parents=True, exist_ok=True)
        else:
            logger.log(f"elaborateDataset: directory output esistente '{out_path}'")

        # Trova e filtra i file .parquet
        pattern = re.compile(filename_regex)
        files = [f for f in ds_path.iterdir() if f.is_file() and pattern.match(f.name)]
        files.sort(key=lambda f: f.name)
        logger.log(f"elaborateDataset: trovati {len(files)} file validi")

        # Elaborazione file
        for file in files:
            logger.log(f"elaborateDataset: inizio elaborazione file '{file.name}'")
            df_res = process_parquet_mocked(str(file), logger, verbosity)
            # Costruisci nome file CSV con timestamp
            ts = datetime.now().strftime("%Y%m%dT%H%M%S")
            base = file.stem
            csv_name = f"{base}_{ts}.csv"
            csv_path = out_path / csv_name
            # Salva in CSV
            df_res.to_csv(csv_path, index=False)
            logger.log(f"elaborateDataset: salvato risultato in '{csv_path}'")

    except Exception as e:
        logger.log(f"elaborateDataset: ERRORE → {e}")
        raise

In [ ]:
elaborateDatasetMocked(
    logger= logger,
    verbosity= 0,
    dataset_path= '/content/drive/MyDrive/Activity/dl-HybridRAG/kaggle/datasets/jjinho/wikipedia-20230701/versions/2',
    filename_regex= '\w*\.parquet',
    output_folder= '/content/drive/MyDrive/Activity/dl-HybridRAG/kaggle/datasets/jjinho/keySeparators.csv')

Test con dataset da Hugging face

In [ ]:
!pip install transformers
!pip install datasets
!pip install tokenizers

In [ ]:
from datasets import load_dataset

In [ ]:
!pip install huggingface_hub

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from huggingface_hub import snapshot_download

In [ ]:
repo_id = "HuggingFaceFW/clean-wikipedia"
directory_name = "en"
download_path = load_dataset(repo_id=repo_id, local_dir='/content/drive/MyDrive/Activity/dl-HybridRAG/huggingFace', allow_patterns="en/.*.parquet")

In [ ]:
wiki = load_dataset("HuggingFaceFW/clean-wikipedia", split="train")

In [ ]:
!pip install -U datasets huggingface_hub fsspec

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from huggingface_hub import snapshot_download
from datasets import get_dataset_config_names
configs = get_dataset_config_names("HuggingFaceFW/clean-wikipedia")

In [ ]:
wiki = load_dataset("HuggingFaceFW/clean-wikipedia", "en", split="train", num_proc=2)

In [ ]:
wiki.to_parquet('/content/drive/MyDrive/Activity/dl-HybridRAG/huggingFace/HuggingFaceFW/wiki-eng-parquet')

In [ ]:
wiki = None

In [ ]:
! pip install pandas

In [ ]:
import pandas as pd
dfWiki = pd.read_parquet('/content/drive/MyDrive/Activity/dl-HybridRAG/huggingFace/HuggingFaceFW/wiki-eng-parquet')

In [ ]:
panda